In [0]:
# Databricks notebook source
# =====================================================
# 1️⃣ Imports & Setup
# =====================================================

import json
import time
from pyspark.sql.functions import current_timestamp, col, max as spark_max
from delta.tables import DeltaTable

## Status and Records Written and Read
status = "SUCCESS"
error_message = None
records_read = 0
records_written = 0

# =====================================================
# 2️⃣ Widgets
# =====================================================

dbutils.widgets.text(
    "table_metadata",
    "{'CustomerProductID':'1', 'TableID': '1', 'SourceTablename': 'Facilities', 'SourceSchema': 'Core', 'CustomerName': 'BostonHealthSystem', 'CustomerCode': 'BOSHOSP', 'ExtractionType': 'Incremental', 'WatermarkColumn': 'LastModifiedDateTime', 'LastExtractWatermark': '1900-01-01 00:00:00', 'TargetSchema': 'Bronze', 'IsActive': 'True'}"
)

dbutils.widgets.text("CustomerCode_Source", "BOSHOSP")

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)
run_id=dbutils.widgets.get("run_id")

## Extract Variables

# Parse JSON safely
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
CustomerCode_Source = (dbutils.widgets.get("CustomerCode_Source"))

print("Table Metadata:", table_metadata)
print("Table CustomerCode_Source:", CustomerCode_Source)

CustomerProductID=int(table_metadata["CustomerProductID"])
TableID = int(table_metadata["TableID"])
SourceTablename = table_metadata["SourceTablename"]
SourceSchema = table_metadata["SourceSchema"].lower()
CustomerName = table_metadata["CustomerName"]
CustomerCode = table_metadata["CustomerCode"].lower()
ExtractionType = table_metadata["ExtractionType"]
WatermarkColumn = table_metadata["WatermarkColumn"]
LastExtractWatermark=table_metadata["LastExtractWatermark"]
TargetSchema=table_metadata["TargetSchema"].lower()
IsActive=table_metadata["IsActive"]

bronze_table = f"clinicalforge.{TargetSchema}.{CustomerCode}_{SourceTablename}"
silver_table = f"clinicalforge.silver.{CustomerCode}_{SourceTablename}"

print(f"Processing Table: {SourceTablename}")
print(bronze_table)
print(silver_table)
print(f"Run ID: {run_id}")

In [0]:
# =====================================================
# 3️⃣ TRY-CATCH BLOCK FOR AUDIT CONTROL
# =====================================================
try:
    # -----------------------------------------
    # Read Bronze
    # -----------------------------------------

    bronze_df = spark.table(bronze_table)

    if ExtractionType in ["Incremental", "MERGE"] and LastExtractWatermark:

        watermark_df = spark.sql(f"""
            SELECT LastExtractWatermark as last_watermark_value
            FROM clinicalforge.metadata.tableslist
            WHERE TableID = {TableID}
        """)

        last_watermark = None
        if watermark_df.count() > 0:
            last_watermark = watermark_df.first()["last_watermark_value"]


    records_read = bronze_df.count()
    print("Records to process:", records_read)


    # -----------------------------------------
    # Add audit columns
    # -----------------------------------------

    bronze_df = (
        bronze_df
        .withColumn("insert_timestamp", current_timestamp())
        .withColumn("update_timestamp", current_timestamp())
    )


    # -----------------------------------------
    # Create Silver if not exists
    # -----------------------------------------

    #spark.sql("create schema if not exists clinicalforge.silver")

    if not spark.catalog.tableExists(silver_table):
        (
            bronze_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )
        records_written = records_read

    else:
        # -------------------------------------
        # FULL
        # -------------------------------------
        if ExtractionType == "FULL":

            (
                bronze_df
                .write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(silver_table)
            )
            records_written = records_read

        # -------------------------------------
        # APPEND
        # -------------------------------------
        elif ExtractionType == "APPEND":

            (
                bronze_df
                .write
                .format("delta")
                .mode("append")
                .saveAsTable(silver_table)
            )
            records_written = records_read

        # -------------------------------------
        # MERGE
        # -------------------------------------
        elif ExtractionType == "Incremental":

            #if not primary_key:
            #    raise ValueError("Primary key required for MERGE.")

            delta_table = DeltaTable.forName(spark, silver_table)

            #merge_condition = f"t.{RowVersion} = s.{RowVersion}"

            (
                delta_table.alias("t")
                .merge(
                    bronze_df.alias("s"),
                    condition="t.RowVersion = s.RowVersion"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

            records_written = records_read
        else:
            raise ValueError("Unsupported load_type")


    # -----------------------------------------
    # Update Watermark (APPEND & MERGE)
    # -----------------------------------------

    if ExtractionType in ["Incremental", "FULL"] and WatermarkColumn:

        max_value = bronze_df.agg(
            spark_max(col(WatermarkColumn))
        ).collect()[0][0]

       
    print("Silver Load Completed Successfully.")


except Exception as e:

    status = "FAILED"
    error_message = str(e)
    print("Error Occurred:", error_message)
    raise


finally:

    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

    # -----------------------------------------
    # Insert into Audit Table
    # -----------------------------------------

    spark.sql(f"""
        UPDATE clinicalforge.metadata.pipelinerun
        SET
            EndDateTime = TIMESTAMP('{end_time}'),
            RunStatus = '{status}',
            RecordsIngested = {records_read}
        WHERE TableID = {TableID} and RunID = {run_id}
    """)

    print("Audit record inserted.")